# WITS Enhanced Inference Pipeline
Inference on Italian WITS dataset with abstraction-aware evaluation metrics and LLM-as-Judge.

**Extension: Semantic Supervision on Italian Wikipedia** - Comparing with original paper on same dataset.

**Metrics:**
- **Abstraction Score**: Measures how much the summary differs from source (1 - n-gram overlap)
- **Compression Ratio**: Generated length / Source length
- **LLM Judge Score**: Llama evaluates quality on 1-5 scale across 4 dimensions

In [1]:
# Clone repository from GitHub (for Kaggle execution)
import os

# Get GitHub token from Kaggle secrets
from kaggle_secrets import UserSecretsClient
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# Repository configuration
REPO_OWNER = "LucaDesderi"  # Replace with your GitHub username
REPO_NAME = "DNLPProj"
BRANCH = "feat/marco"  # Replace with your branch name

# Clone repository with token authentication
!git clone https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git

# Change directory to the notebook location
%cd {REPO_NAME}/extension-italian-wits-semantic/inference

# Checkout specific branch
!git checkout {BRANCH}

print(f"✓ Repository cloned and switched to branch: {BRANCH}")
!pwd

Cloning into 'DNLPProj'...
remote: Write access to repository not granted.
fatal: unable to access 'https://github.com/LucaDesderi/DNLPProj.git/': The requested URL returned error: 403
[Errno 2] No such file or directory: 'DNLPProj/extension-italian-wits-semantic/inference'
/kaggle/working
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
✓ Repository cloned and switched to branch: feat/marco
/kaggle/working


## 1. Setup

In [2]:
# Install deps
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10' matplotlib seaborn

!python -m spacy download it_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 110.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
import warnings
import logging

# Suppress warning messages
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)
import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

# Load spacy for sentence segmentation
nlp = spacy.load("it_core_news_sm")

## 2. Configuration

In [4]:
# Auth
login(token=HF_TOKEN)

# ArXiv model configuration
SIGEXT_CONFIG = {
    "model_id": "LookUpMark/sigext-wits-it-10k-060t",
    "skip_samples": 25000,
    "threshold": 0.60
}

# Use 8-bit for best efficiency
QUANT_CONFIG = {
    "load_in_8bit": True
}

GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "num_test_samples": 100,
    "max_length": 2048,
    "output_dir": "./results_enhanced"
}

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Testing with {GLOBAL_CONFIG['num_test_samples']} samples")

Testing with 100 samples


## 3. Enhanced Prompts

In [5]:
# Italian WITS summarization prompt - Anti-Hallucination Version
SUMMARY_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Sei un riassuntore FEDELE di articoli enciclopedici. I tuoi riassunti devono contenere SOLO fatti presenti nella fonte.

REGOLE CRITICHE DI FEDELTÀ:
1. Usa SOLO informazioni esplicitamente presenti nel testo fornito.
2. NON aggiungere conoscenze esterne, date, fatti o nomi non presenti nella fonte.
3. Se la fonte non menziona qualcosa, NON devi menzionarlo nemmeno tu.
4. In caso di dubbio su un fatto, OMETTILO piuttosto che inventarlo.
5. NON inferire o estrapolare oltre quanto scritto.

REGOLE DI SCRITTURA:
1. Scrivi UN SOLO paragrafo (50-80 parole). NO titoli, NO elenchi.
2. SINTETIZZA e RIFORMULA - mai copiare frasi letteralmente.
3. Inizia con una definizione o contestualizzazione del soggetto.
<|eot_id|><|start_header_id|>user<|end_header_id|>
TESTO FONTE:
{source}

CONCETTI CHIAVE DA INTEGRARE (tutti dalla fonte sopra):
{keyphrases}

Scrivi un riassunto FEDELE usando SOLO fatti dalla fonte sopra. Non aggiungere conoscenze esterne.<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""




# LLM-as-Judge prompt - SOURCE-AWARE Version (Full Source)
JUDGE_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert summary evaluator. You have access to the full SOURCE document and the REFERENCE summary.
Evaluate if the GENERATED summary is faithful to the source content.

IMPORTANT: The generated summary may include information from the SOURCE that is NOT in the reference.
This is acceptable as long as all information is verifiable in the SOURCE.

Respond ONLY with valid JSON. Use double quotes. Max 15 words per reason.
<|eot_id|><|start_header_id|>user<|end_header_id|>

SOURCE DOCUMENT:
{source}

REFERENCE SUMMARY:
{reference}

GENERATED SUMMARY:
{generated}

---
Rate (1-5 each):
1. FAITHFULNESS: All facts verifiable in SOURCE? (1=fabricated, 5=all verified)
2. COMPLETENESS: Covers main points? (1=missing, 5=complete)
3. CONCISENESS: Fluid, not repetitive? (1=verbose, 5=concise)
4. ABSTRACTION: Rephrases vs copies? (1=verbatim, 5=novel phrasing)

{{"faithfulness": X, "faithfulness_reason": "...", "completeness": X, "completeness_reason": "...", "conciseness": X, "conciseness_reason": "...", "abstraction": X, "abstraction_reason": "..."}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

## 4. Helper Functions

In [6]:
def get_test_data(skip_samples, num_samples):
    print(f"  Loading test data (skipping {skip_samples})...")
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    dataset = dataset.skip(skip_samples)
    
    test_data = []
    for entry in dataset:
        source = entry['source']
        summary = entry['summary']
        
        if len(source) < 500 or len(summary) < 50 or len(source) > 10000:
            continue
            
        test_data.append({"source": source, "reference": summary})
        
        if len(test_data) >= num_samples:
            break
    
    print(f"  Test data ready: {len(test_data)} samples")
    return test_data


def load_sigext_model(model_id):
    print(f"  Loading SigExt: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to("cuda")
    return model, tokenizer


def load_llm(model_id, quant_config):
    quant_name = "4-bit" if "load_in_4bit" in quant_config else "8-bit"
    print(f"  Loading LLM ({quant_name}): {model_id}...")
    
    bnb_config = BitsAndBytesConfig(**quant_config)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer


def extract_salient_sentences(text, model, tokenizer, max_length):
    sentences = [sent.text.strip() for sent in nlp(text).sents if len(sent.text.strip()) > 20]
    
    if not sentences:
        return [], ""
    
    salient_sentences = []
    
    for sent in sentences:
        inputs = tokenizer(
            sent,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to("cuda")
        
        with torch.no_grad():
            logits = model(**inputs).logits
        
        preds = torch.argmax(logits, dim=2)[0].tolist()
        
        valid_preds = preds[1:-1] if len(preds) > 2 else preds
        if valid_preds:
            salient_ratio = sum(valid_preds) / len(valid_preds)
            if salient_ratio > 0.5:
                salient_sentences.append(sent)
    
    keyphrases_text = "\n".join(f"- {s}" for s in salient_sentences)
    
    return salient_sentences, keyphrases_text


def preprocess_dataset(test_data, sigext_model, sigext_tokenizer, max_length):
    processed_data = []
    for item in tqdm(test_data, desc="    Extracting Salient Sentences"):
        salient_sents, keys_text = extract_salient_sentences(
            item['source'], sigext_model, sigext_tokenizer, max_length
        )
        processed_data.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": salient_sents,
            "keyphrases": keys_text
        })
    return processed_data


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

clear_gpu_memory()

## 5. New Abstraction Metrics

In [7]:
def get_ngrams(text, n=3):
    """Extract n-grams from text."""
    words = text.lower().split()
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]


def compute_abstraction_score(source, generated, n=3):
    """
    Compute abstraction score: 1 - (n-gram overlap with source).
    Higher = more abstractive (less copying).
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)
    
    if not gen_ngrams:
        return 1.0  # Empty summary = no copying
    
    copied = sum(1 for ng in gen_ngrams if ng in source_ngrams)
    copy_ratio = copied / len(gen_ngrams)
    
    return 1.0 - copy_ratio


def compute_compression_ratio(source, generated):
    """Compute compression ratio (lower = more compressed)."""
    if len(source) == 0:
        return 1.0
    return len(generated) / len(source)


def compute_novel_ngrams(source, generated, n=2):
    """
    Compute percentage of n-grams in generated that are NOT in source.
    Higher = more novel content.
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)
    
    if not gen_ngrams:
        return 0.0
    
    novel = sum(1 for ng in gen_ngrams if ng not in source_ngrams)
    return novel / len(gen_ngrams)

## 6. LLM-as-Judge Evaluation

In [8]:
def parse_judge_response(text):
    """Robust JSON parsing for LLM judge responses."""
    import re
    import json
    
    KEYS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']
    
    # Method 1: Try direct JSON parsing
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        json_str = json_match.group()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass
    
    # Method 2: Clean common issues and retry
    if json_match:
        json_str = json_match.group()
        json_str = json_str.replace("'", '"')
        json_str = re.sub(r',\s*}', '}', json_str)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass
    
    # Method 3: Extract individual values with regex
    scores = {}
    for key in KEYS:
        match = re.search(rf'"{key}"\s*:\s*(\d)', text, re.IGNORECASE)
        if match:
            scores[key] = int(match.group(1))
        reason_match = re.search(rf'"{key}_reason"\s*:\s*"([^"]*)"', text)
        if reason_match:
            scores[f"{key}_reason"] = reason_match.group(1)
    
    return scores


def llm_judge_evaluate(source, generated, reference, judge_chain):
    """Use LLM to judge quality of generated summary against source."""
    ENGLISH_KEYS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']
    DEFAULT_SCORES = {k: 3 for k in ENGLISH_KEYS}
    DEFAULT_SCORES.update({f"{k}_reason": "Unable to evaluate" for k in ENGLISH_KEYS})
    
    try:
        result = judge_chain.invoke({
            "source": source,
            "reference": reference,
            "generated": generated
        })
        
        result_text = result.split("assistant<|end_header_id|>")[-1].strip()
        scores = parse_judge_response(result_text)
        
        for key in ENGLISH_KEYS:
            if key not in scores:
                scores[key] = 3
            else:
                scores[key] = max(1, min(5, int(scores[key])))
            reason_key = f"{key}_reason"
            if reason_key not in scores:
                scores[reason_key] = ""
        
        return scores
            
    except Exception as e:
        print(f"      Judge error: {e}")
        return DEFAULT_SCORES.copy()


## 7. Enhanced Evaluation Function

In [9]:
def run_enhanced_evaluation(processed_data, summary_chain, judge_chain):
    """Run evaluation with traditional + abstraction + LLM-judge metrics."""
    
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    
    metrics = {
        # Traditional
        "bert": [], "rouge1": [], "kir": [],
        # Abstraction
        "abstraction": [], "compression": [], "novel_ngrams": [],
        # LLM Judge
        "judge_faithfulness": [], "judge_completeness": [], 
        "judge_conciseness": [], "judge_abstraction": []
    }
    samples = []
    
    for item in tqdm(processed_data, desc="    Generating & Evaluating"):
        try:
            keys_text = item['keyphrases']
            salient_sents = item['salient_sentences']
            
            # Generate summary
            res = summary_chain.invoke({"source": item['source'], "keyphrases": keys_text})
            gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()
            
            # === TRADITIONAL METRICS ===
            
            # ROUGE-1 and ROUGE-L
            rouge_scores = scorer.score(item['reference'], gen_summary)
            metrics["rouge1"].append(rouge_scores['rouge1'].fmeasure)
                        
            # BERTScore (Italian)
            _, _, F1 = bert_score([gen_summary], [item['reference']], lang="it", verbose=False)
            bert_sc = F1.mean().item()
            metrics["bert"].append(bert_sc)
            
            # KIR
            kir_score = 0.0
            if salient_sents:
                gen_lower = gen_summary.lower()
                hits = 0
                for sent in salient_sents:
                    words = [w.lower() for w in sent.split() if len(w) > 4]
                    if words:
                        word_hits = sum(1 for w in words if w in gen_lower)
                        if word_hits / len(words) > 0.3:
                            hits += 1
                kir_score = hits / len(salient_sents)
            metrics["kir"].append(kir_score)
            
            # === ABSTRACTION METRICS ===
            
            abstraction = compute_abstraction_score(item['source'], gen_summary)
            compression = compute_compression_ratio(item['source'], gen_summary)
            novel = compute_novel_ngrams(item['source'], gen_summary)
            
            metrics["abstraction"].append(abstraction)
            metrics["compression"].append(compression)
            metrics["novel_ngrams"].append(novel)
            
            # === LLM JUDGE ===
            
            judge_scores = llm_judge_evaluate(
                item["source"], gen_summary, item["reference"], judge_chain
            )
            
            metrics["judge_faithfulness"].append(judge_scores["faithfulness"])
            metrics["judge_completeness"].append(judge_scores["completeness"])
            metrics["judge_conciseness"].append(judge_scores["conciseness"])
            metrics["judge_abstraction"].append(judge_scores["abstraction"])
            
            # Store sample details
            samples.append({
                "source": item['source'][:500] + "..." if len(item['source']) > 500 else item['source'],
                "reference": item['reference'],
                "salient_sentences": salient_sents,
                "generated_summary": gen_summary,
                "scores": {
                    "bert": float(bert_sc),
                    "rouge1": float(rouge_scores['rouge1'].fmeasure),
                                        "kir": float(kir_score),
                    "abstraction": float(abstraction),
                    "compression": float(compression),
                    "novel_ngrams": float(novel),
                    "judge": judge_scores
                }
            })
                
        except Exception as e:
            print(f"    Error: {e}")
            continue
    
    return metrics, samples


## 8. Main Evaluation Loop

In [10]:
print("="*60)
print("PHASE 1: LOADING MODELS")
print("="*60)

# Load SigExt
sigext_model, sigext_tokenizer = load_sigext_model(SIGEXT_CONFIG["model_id"])

# Load test data
test_data = get_test_data(
    SIGEXT_CONFIG["skip_samples"],
    GLOBAL_CONFIG["num_test_samples"]
)

# Extract salient sentences
processed_data = preprocess_dataset(
    test_data,
    sigext_model,
    sigext_tokenizer,
    GLOBAL_CONFIG["max_length"]
)

# Cleanup SigExt
del sigext_model, sigext_tokenizer
clear_gpu_memory()

print("\n" + "="*60)
print("PHASE 2: LOADING LLM")
print("="*60)

# Load LLM
llm_model, llm_tokenizer = load_llm(
    GLOBAL_CONFIG["llm_model_id"], 
    QUANT_CONFIG
)

# Create text generation pipeline
gen_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=256,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# Create chains
summary_prompt = PromptTemplate(
    template=SUMMARY_PROMPT,
    input_variables=["source", "keyphrases"]
)
summary_chain = summary_prompt | llm | StrOutputParser()

judge_prompt = PromptTemplate(
    template=JUDGE_PROMPT, 
    input_variables=["source", "reference", "generated"]
)
judge_chain = judge_prompt | llm | StrOutputParser()

print("\n" + "="*60)
print("PHASE 3: RUNNING EVALUATION")
print("="*60)

metrics, samples = run_enhanced_evaluation(processed_data, summary_chain, judge_chain)

print("\n" + "="*60)
print("EVALUATION COMPLETE!")
print("="*60)

PHASE 1: LOADING MODELS
  Loading SigExt: LookUpMark/sigext-wits-it-10k-060t...
  Loading test data (skipping 25000)...


  Test data ready: 100 samples


    Extracting Salient Sentences:   0%|          | 0/100 [00:00<?, ?it/s]


PHASE 2: LOADING LLM
  Loading LLM (8-bit): meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


EVALUATION COMPLETE!


## 9. Results Summary

In [12]:
# Compute and display results
results = {
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": SIGEXT_CONFIG["model_id"],
        "llm_model": GLOBAL_CONFIG["llm_model_id"],
        "num_samples": len(samples),
        "prompt_type": "optimized_abstractive_v2"
    },
    "metrics": {}
}

# Aggregate metrics
for m in ["bert", "rouge1", "kir", "abstraction", "compression", "novel_ngrams"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

for m in ["judge_faithfulness", "judge_completeness", "judge_conciseness", "judge_abstraction"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

if metrics.get("judge_faithfulness"):
    overall = [sum(metrics[f"judge_{k}"][i] for k in ["faithfulness", "completeness", "conciseness", "abstraction"]) 
               for i in range(len(metrics["judge_faithfulness"]))]
    results["metrics"]["judge_overall"] = {"mean": float(np.mean(overall)) / 4, "std": float(np.std(overall)) / 4}

results["samples"] = samples

# Print summary
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

m = results["metrics"]
if "bert" in m:
    print(f"\nTraditional Metrics:")
    print(f"  BERT Score:  {m['bert']['mean']:.4f} +/- {m['bert']['std']:.4f}")
    print(f"  ROUGE-1:     {m['rouge1']['mean']:.4f} +/- {m['rouge1']['std']:.4f}")
    print(f"  KIR:         {m['kir']['mean']:.2%}")

if "abstraction" in m:
    print(f"\nAbstraction Metrics:")
    print(f"  Abstraction: {m['abstraction']['mean']:.4f}")
    print(f"  Novel:       {m['novel_ngrams']['mean']:.2%}")
    print(f"  Compression: {m['compression']['mean']:.2%}")

if "judge_faithfulness" in m:
    print(f"\nLLM Judge (1-5):")
    print(f"  Faithfulness: {m['judge_faithfulness']['mean']:.2f}")
    print(f"  Completeness: {m['judge_completeness']['mean']:.2f}")
    print(f"  Conciseness:  {m['judge_conciseness']['mean']:.2f}")
    print(f"  Abstraction:  {m['judge_abstraction']['mean']:.2f}")
    print(f"  Overall:      {m['judge_overall']['mean']:.2f}")

# Save
output_file = os.path.join(GLOBAL_CONFIG["output_dir"], "results_enhanced.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {output_file}")


RESULTS SUMMARY

Traditional Metrics:
  BERT Score:  0.6631 +/- 0.0398
  ROUGE-1:     0.2116 +/- 0.0924
  KIR:         45.87%

Abstraction Metrics:
  Abstraction: 0.6554
  Novel:       49.97%
  Compression: 24.78%

LLM Judge (1-5):
  Faithfulness: 4.80
  Completeness: 4.72
  Conciseness:  4.72
  Abstraction:  4.65
  Overall:      4.72

Saved: ./results_enhanced/results_enhanced.json


## 10. Sample Analysis

In [13]:
# Sample outputs
print("=" * 60)
print("SAMPLE OUTPUTS")
print("=" * 60)

for idx in [0, len(samples)//2, len(samples)-1]:
    s = samples[idx]
    print(f"\n[Sample {idx}]")
    print(f"Reference: {s['reference'][:200]}...")
    print(f"Generated: {s['generated_summary'][:200]}...")
    sc = s['scores']
    print(f"Scores: BERT={sc['bert']:.2f} ROUGE={sc['rouge1']:.2f} KIR={sc['kir']:.2f} Abstr={sc['abstraction']:.2f}")


SAMPLE OUTPUTS

[Sample 0]
Reference: 


'''''1,039/Smoothed Out Slappy Hours''''' è una compilation della punk rock band statunitense Green Day, pubblicata nel 1991 dalla Lookout! Records.
...
Generated: La compilation ''1,039/Smoothed Out Slappy Hours'' raccoglie i pezzi del primo album ''39/Smooth'', dei due EP ''1,000 Hours'' e ''Slappy'', e l'inedita ''I Want to Be Alone''. Pubblicata dalla Lookou...
Scores: BERT=0.73 ROUGE=0.26 KIR=0.46 Abstr=0.51

[Sample 50]
Reference: La '''lettera di credito''' è un documento, emesso da un istituto di credito, che funge allo stesso tempo da garanzia affinché un soggetto possa ottenere un finanziamento da parte di altri soggetti, c...
Generated: La lettera di credito è un mezzo di pagamento nato nel medioevo per evitare di dover portare denaro e preziosi durante i lunghi viaggi. L'acquirente che emette la lettera di credito richiede documenti...
Scores: BERT=0.69 ROUGE=0.29 KIR=0.71 Abstr=0.62

[Sample 99]
Reference: 


Costruttore di bassi e 

## 11. Cleanup

In [15]:
# Cleanup
del llm_model, llm_tokenizer, gen_pipe, llm
clear_gpu_memory()

print(" Cleanup complete!")